# 17. Gravitational Wave Data Analysis

Matched filtering, parameter estimation with bilby, population inference.

In [ ]:
import bilby
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import chirp
print('bilby', bilby.__version__)

## 17.1 Synthetic GW signal (inspiral)

In [ ]:
fs = 4096  # Hz
duration = 4.0
t = np.arange(0, duration, 1/fs)
# TaylorF2-like chirp
f0, f1 = 20., 500.
h = chirp(t, f0=f0, f1=f1, t1=duration, method='linear') * np.exp(-t*50)
plt.plot(t[:2000], h[:2000])
plt.xlabel('Time (s)'); plt.ylabel('Strain'); plt.title('Synthetic inspiral'); plt.show()

## 17.2 Matched filtering

In [ ]:
from scipy.signal import correlate
# Template
temp = chirp(t, f0=f0, f1=f1, t1=duration, method='linear')
snr = correlate(h, temp, mode='same')
plt.plot(t, snr)
plt.xlabel('Time (s)'); plt.ylabel('SNR'); plt.title('Matched filter output'); plt.show()

## 17.3 Bilby parameter estimation (BBH)

In [ ]:
import bilby
from bilby.gw.detector import InterferometerList
# Setup
ifos = InterferometerList(['H1', 'L1', 'V1'])
for ifo in ifos:
    ifo.set_strain_data_from_power_spectral_densities(
        sampling_frequency=fs, duration=duration,
        start_time=0)

# Priors
priors = bilby.gw.prior.BBHPriorDict()
priors['mass_1'] = bilby.prior.Uniform(5, 50, 'mass_1')
priors['mass_2'] = bilby.prior.Uniform(5, 50, 'mass_2')
priors['a_1'] = bilby.prior.Uniform(0, 0.99)
priors['a_2'] = bilby.prior.Uniform(0, 0.99)
print('Priors set:', list(priors.keys())[:8])

## 17.4 Population inference (mock)

In [ ]:
# Population model: mass distribution p(m) ~ m^-2.35
# Mock posterior samples
np.random.seed(42)
masses = np.random.power(2.35, 1000) * 50 + 5
plt.hist(masses, bins=40, density=True, alpha=0.7)
plt.xlabel('Mass (Msun)'); plt.ylabel('p(m)'); plt.title('Mock BH mass distribution'); plt.show()